In [ ]:
import nbformat as nbf

nb = nbf.v4.new_notebook()
cells = []

def md(text):
    cells.append(nbf.v4.new_markdown_cell(text))

def code(text):
    cells.append(nbf.v4.new_code_cell(text))

md("# Breast Cancer Classification using K-Nearest Neighbors (KNN)\n"
   "**AI-ML Assignment 4**\n\n"
   "Dataset: [Breast Cancer Wisconsin Diagnostic Dataset]"
   "(https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data)")

md("## Task 1: Data Understanding\n"
   "1. Load the dataset using Pandas\n"
   "2. Display the first five records\n"
   "3. Identify numerical features and the target variable\n"
   "4. Display dataset information and summary statistics")

code("""import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

df = pd.read_csv("breast_cancer.csv")
df.head()""")

code("""numerical_features = [col for col in df.columns if col not in ["id", "diagnosis"]]
target_variable = "diagnosis"

print("Number of numerical features:", len(numerical_features))
print("Target variable:", target_variable)""")

code("""df.info()""")

code("""df.describe()""")

md("## Task 2: Data Preprocessing\n"
   "- Check for missing values\n"
   "- Remove unnecessary columns (if any)\n"
   "- Encode the target variable\n"
   "- Normalize/standardize the feature values\n"
   "- Split the dataset into 80% training and 20% testing")

code("""print(df.isnull().sum().sum(), "total missing values")""")

code("""# "id" is just a row identifier, not a predictive feature
df = df.drop("id", axis=1)

# Encode the target variable (M = Malignant -> 1, B = Benign -> 0)
le = LabelEncoder()
df["diagnosis"] = le.fit_transform(df["diagnosis"])
print("Target encoding:", dict(zip(le.classes_, le.transform(le.classes_))))""")

code("""X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features - essential for KNN since it is a distance-based algorithm
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set size:", X_train.shape[0])
print("Testing set size :", X_test.shape[0])""")

md("## Task 3: Model Development\n"
   "1. Train a K-Nearest Neighbors classifier\n"
   "2. Use K = 5 as the initial value\n"
   "3. Predict the class labels for the test dataset")

code("""knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

pd.DataFrame({"Actual": y_test.values[:10], "Predicted": y_pred[:10]})""")

md("## Task 4: Model Evaluation\n"
   "Evaluate using Accuracy, Precision, Recall, F1-Score, and a Confusion Matrix")

code("""accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy Score : {accuracy:.4f}")
print(f"Precision      : {precision:.4f}")
print(f"Recall         : {recall:.4f}")
print(f"F1-Score       : {f1:.4f}")""")

code("""cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Benign", "Malignant"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - Breast Cancer Classification (KNN, K=5)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()""")

md("**Observations:**\n\n"
   "1. The model achieves a high accuracy of about 95.6%, showing that KNN with K=5 "
   "separates malignant and benign tumors very well once features are standardized.\n"
   "2. Precision (about 97.4%) is slightly higher than recall (about 90.5%) for the "
   "malignant class, meaning the few errors the model makes are more often missed "
   "malignant cases (false negatives) than false alarms - the more clinically "
   "concerning type of error.\n"
   "3. Because KNN relies purely on distance between points, its strong performance "
   "here depends heavily on the standardization step; without it, features measured "
   "on larger scales (like area) would dominate the distance calculation and degrade "
   "accuracy.")

md("## Task 5: Conclusion\n\n"
   "This project built a K-Nearest Neighbors (K=5) classifier to distinguish "
   "malignant from benign breast tumors using diagnostic measurements, achieving an "
   "accuracy of 0.96, precision of 0.97, recall of 0.90, and F1-score of 0.94 on the "
   "test set. Feature scaling was essential here, since KNN classifies points based "
   "on distance to their nearest neighbors, and unscaled features measured on very "
   "different ranges (such as area versus smoothness) would let larger-magnitude "
   "features dominate the distance calculation and distort predictions. After "
   "standardizing all features, the model separated malignant and benign cases with "
   "high accuracy, confirming that these diagnostic measurements are strong "
   "discriminators of tumor type. A key limitation of KNN is that it is "
   "computationally expensive at prediction time, since it must compute distances to "
   "all training points for every new prediction, making it slow to scale to very "
   "large datasets compared to models that learn a fixed set of parameters upfront.")

nb['cells'] = cells
with open("Assignment-4.ipynb", "w") as f:
    nbf.write(nb, f)

print("Notebook created.")